In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

ROOT = Path(".").resolve().parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "steady_state_detection"))

PATH    = ROOT / "data" / "real-life" / "helpdesk.xes"
CACHE   = ROOT / "cache"
RESULTS = ROOT / "results"

CACHE.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

print("Data file exists:", PATH.exists())

In [ ]:

import pm4py

log = pm4py.read_xes(str(PATH))
print(f"Events: {len(log):,}")
print(f"Cases:  {log['case:concept:name'].nunique():,}")
print(f"Period: {log['time:timestamp'].min()} → {log['time:timestamp'].max()}")
log.head()

In [ ]:
from time_series_creation import *
avg_cc=create_concurrent_cases_timeseries(log)
avg_tt=create_avg_throughtput_time_timeseries(log)

In [ ]:
from time_series_preprocessing import *
from setttings import *
_cc_pct = trim_tail_pct(avg_cc, pct=0.10)
_cc_mag = trim_tail_magnitude(avg_cc, k=1.5, window=7)
_cc_pk  = trim_tail_peak(avg_cc, frac=0.60, window=7)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax = axes[0]
ax.plot(avg_cc.index, avg_cc, color="steelblue", alpha=0.4, label="original")
ax.plot(_cc_pct.index, _cc_pct, color="darkorange", label=f"pct (−10 %, n={len(_cc_pct)})")
ax.plot(_cc_mag.index, _cc_mag, color="green",      label=f"magnitude (n={len(_cc_mag)})")
ax.plot(_cc_pk.index,  _cc_pk,  color="purple",     label=f"peak×0.60 (n={len(_cc_pk)})")
for s, c in [(_cc_pct, "darkorange"), (_cc_mag, "green"), (_cc_pk, "purple")]:
    ax.axvline(s.index[-1], linestyle="--", color=c, linewidth=1)
ax.set_title("Concurrent Cases — trim method comparison")
ax.legend()

ax = axes[1]
ax.plot(avg_cc.index, avg_cc, color="steelblue", alpha=0.4, label="original")
ax.axhline(0.60 * avg_cc.max(), linestyle=":", color="purple",  linewidth=1.2, label=f"peak threshold (×0.60 = {0.60*avg_cc.max():.1f})")
ax.axhline(0.50 * avg_cc.max(), linestyle=":", color="red",     linewidth=1.2, label=f"peak threshold (×0.50 = {0.50*avg_cc.max():.1f})")
ax.axhline(0.70 * avg_cc.max(), linestyle=":", color="teal",    linewidth=1.2, label=f"peak threshold (×0.70 = {0.70*avg_cc.max():.1f})")
ax.set_title("Peak-fraction thresholds — tune `frac` here")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
TRIM_METHOD = "peak"
TRIM_PCT    = 0.25
TRIM_K      = 1.5
TRIM_FRAC   = 0.70
TRIM_WINDOW = 7

set_global_seed(1904)
cfg = Split3WayConfig(train_frac=0.7, val_frac=0.1, test_frac=0.2)

avg_cc_trim = apply_trim(avg_cc, TRIM_METHOD, pct=TRIM_PCT, k=TRIM_K, frac=TRIM_FRAC, window=TRIM_WINDOW)
canonical_end = avg_cc_trim.index[-1]
_, _, _, canonical_train_split, canonical_val_split = split_timeseries(avg_cc_trim, cfg)

print(f"trim_end    : {canonical_end.date()}")
print(f"train_split : {canonical_train_split.date()}")
print(f"val_split   : {canonical_val_split.date()}")

avg_tt_trim = avg_tt[avg_tt.index <= canonical_end]

print(f"CC: {len(avg_cc)} → {len(avg_cc_trim)}  (dropped {len(avg_cc) - len(avg_cc_trim)} rows)")
print(f"TT: {len(avg_tt)} → {len(avg_tt_trim)}  (dropped {len(avg_tt) - len(avg_tt_trim)} rows)")

train_cc, val_cc, test_cc = split_by_dates(avg_cc_trim, canonical_train_split, canonical_val_split)
train_tt, val_tt, test_tt = split_by_dates(avg_tt_trim, canonical_train_split, canonical_val_split)

print(f"CC  — train: {len(train_cc)} ({train_cc.index[0].date()} → {train_cc.index[-1].date()})")
print(f"       val : {len(val_cc)} ({val_cc.index[0].date()} → {val_cc.index[-1].date()})")
print(f"      test : {len(test_cc)} ({test_cc.index[0].date()} → {test_cc.index[-1].date()})")
print(f"TT  — train: {len(train_tt)} ({train_tt.index[0].date()} → {train_tt.index[-1].date()})")
print(f"       val : {len(val_tt)} ({val_tt.index[0].date()} → {val_tt.index[-1].date()})")
print(f"      test : {len(test_tt)} ({test_tt.index[0].date()} → {test_tt.index[-1].date()})")

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(train_cc.index, train_cc, label="train")
ax.plot(val_cc.index,   val_cc,   label="val")
ax.plot(test_cc.index,  test_cc,  label="test")
ax.set_title(f"Concurrent Cases — split after trim (method={TRIM_METHOD!r})")
ax.set_ylabel("Active cases")
ax.legend()
plt.tight_layout()
plt.show()
plt.close()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(train_tt.index, train_tt, label="train")
ax.plot(val_tt.index,   val_tt,   label="val")
ax.plot(test_tt.index,  test_tt,  label="test")
ax.set_title(f"Throughput Time — split after trim (method={TRIM_METHOD!r})")
ax.set_ylabel("Avg TT")
ax.legend()
plt.tight_layout()
plt.show()

## Forecasting — Naive / ETS / SARIMAX

In [ ]:
from time_series_prediction import (
    forecast_naive, forecast_naive_recent, forecast_seasonal_naive,
    forecast_ets, forecast_sarimax, forecast_ridge, forecast_gru, forecast_nbeats,
    tune_on_val, run_pipeline,
    SEASONAL_NAIVE_GRID, ETS_GRID, SARIMAX_GRID, RIDGE_GRID, GRU_GRID, NBEATS_GRID,
)
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import numpy as np

print(f"Seasonal naive grid: {len(SEASONAL_NAIVE_GRID)} candidates")
print(f"ETS grid:            {len(ETS_GRID)} candidates")
print(f"SARIMAX grid:        {len(SARIMAX_GRID)} candidates")
print(f"Ridge grid:          {len(RIDGE_GRID)} candidates")
print(f"GRU grid:            {len(GRU_GRID)} candidates")
print(f"N-BEATS grid:        {len(NBEATS_GRID)} candidates")

In [ ]:
COLORS = {
    "naive":          "red",
    "naive_recent":   "salmon",
    "seasonal_naive": "darkorange",
    "ets":            "purple",
    "sarimax":        "saddlebrown",
    "ridge":          "steelblue",
    "gru":            "darkgreen",
    "nbeats":         "deeppink",
}

def _trim_dir_name(trim_method, trim_pct, trim_k, trim_frac):
    """Return a folder name encoding the active trim method and its key parameter."""
    if trim_method is None:
        return "none"
    if trim_method == "pct":
        return f"pct_{trim_pct:g}"
    if trim_method == "magnitude":
        return f"magnitude_{trim_k:g}"
    if trim_method == "peak":
        return f"peak_{trim_frac:g}"
    return str(trim_method)

def plot_timeseries_splits(raw, trimmed, train, val, test, title, save_path=None):
    """Plot the four parts of a time series: train / val / test / truncated tail."""
    truncated = raw.loc[raw.index > trimmed.index[-1]]
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(train.index,     train,     color="steelblue", label="train")
    ax.plot(val.index,       val,       color="orange",    label="val")
    ax.plot(test.index,      test,      color="green",     label="test")
    if len(truncated):
        ax.plot(truncated.index, truncated, color="gray", linestyle="--",
                alpha=0.7, label="truncated")
        ax.axvline(truncated.index[0], linestyle=":", color="gray", linewidth=1)
    ax.axvline(train.index[-1], linestyle=":", color="gray", linewidth=1)
    ax.axvline(val.index[-1],   linestyle=":", color="gray", linewidth=1)
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close()
    else:
        plt.show()

def plot_forecasts(train, val, test, preds, title, save_path=None):
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(train.index, train, color="steelblue", label="train")
    ax.plot(val.index,   val,   color="orange",    label="val")
    ax.plot(test.index,  test,  color="green",     label="test (true)", linewidth=1.5)
    for name, yhat in preds.items():
        ax.plot(test.index, yhat, linestyle="--", color=COLORS.get(name, "black"), label=name)
    ax.axvline(train.index[-1], linestyle=":", color="gray", linewidth=1)
    ax.axvline(val.index[-1],   linestyle=":", color="gray", linewidth=1)
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close()
    else:
        plt.show()


## Batch evaluation — all datasets

In [ ]:
REAL_TRIM_CONFIGS = [
    dict(trim_method="ssd"),
    dict(trim_method=None),
    dict(trim_method="peak", trim_frac=0.6),
    dict(trim_method="peak", trim_frac=0.7),
    dict(trim_method="peak", trim_frac=0.8),
    dict(trim_method="magnitude", trim_k=1.0),
    dict(trim_method="magnitude", trim_k=2.0),
    dict(trim_method="magnitude", trim_k=3.0),
]

In [ ]:
import sys
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import pm4py
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm.auto import tqdm
from time_series_preprocessing import *
from setttings import *
from time_series_creation import (
    create_concurrent_cases_timeseries,
    create_avg_throughtput_time_timeseries,
)
from time_series_preprocessing import apply_trim, split_timeseries, split_by_dates
from time_series_prediction import run_pipeline

sys.path.insert(0, str(ROOT / "analysis"))
from error_direction import _write_ts_prediction_series


def evaluate_all_datasets(
    data_dir: str | Path,
    results_dir: str | Path,
    trim_method: str = "pct",
    trim_pct: float = 0.15,
    trim_k: float = 1.5,
    trim_frac: float = 0.60,
    trim_window: int = 7,
    train_frac: float = 0.7,
    val_frac: float = 0.1,
    test_frac: float = 0.2,
    models: list[str] | None = None,
    tuning: str = "small",
    include_naive_recent: bool = False,
    save_plots: bool = True,
) -> pd.DataFrame:
    """Run the full forecasting pipeline on every .xes file in data_dir."""
    data_dir    = Path(data_dir)
    trim_name   = _trim_dir_name(trim_method, trim_pct, trim_k, trim_frac)
    results_dir = Path(results_dir) / trim_name
    results_dir.mkdir(parents=True, exist_ok=True)

    cfg     = Split3WayConfig(train_frac=train_frac, val_frac=val_frac, test_frac=test_frac)
    trim_kw = dict(pct=trim_pct, k=trim_k, frac=trim_frac, window=trim_window)

    if include_naive_recent:
        base = models if models is not None else [
            "naive", "seasonal_naive", "ets", "sarimax", "theta", "stl",
            "ridge", "ridge_mimo", "gru", "gru_mimo", "nbeats", "nhits", "tft",
        ]
        models_to_run = base + ["naive_recent"]
    else:
        models_to_run = models

    all_rows: list[dict] = []
    all_timing: list[pd.DataFrame] = []
    xes_files = sorted(data_dir.glob("*.xes"))
    tqdm.write(f"Found {len(xes_files)} dataset(s): {[f.stem for f in xes_files]}")
    tqdm.write(f"Saving to: {results_dir}\n")

    series_map = {
        "concurrent_cases": create_concurrent_cases_timeseries,
        "throughput_time":  create_avg_throughtput_time_timeseries,
    }

    dataset_bar = tqdm(xes_files, desc="Datasets", unit="dataset", leave=True)
    for xes_path in dataset_bar:
        dataset_name = xes_path.stem
        dataset_bar.set_description(f"Datasets | {dataset_name}")

        try:
            log = pm4py.read_xes(str(xes_path))
        except Exception:
            tqdm.write(f"  [SKIP] failed to load {dataset_name}:\n{traceback.format_exc()}")
            continue

        out_dir = results_dir / dataset_name
        out_dir.mkdir(parents=True, exist_ok=True)
        dataset_rows: list[dict] = []
        dataset_timing: list[pd.DataFrame] = []

        _cc_raw     = create_concurrent_cases_timeseries(log, plot=False)
        if trim_method == "ssd":
            from ssd_trim import run_ssd_trim
            _ssd_result = run_ssd_trim(log, window_step="D")
            canonical_end = _ssd_result["cutoff"] if _ssd_result["cutoff"] is not None else _cc_raw.index[-1]
            _cc_trimmed = _cc_raw[_cc_raw.index <= canonical_end]
        else:
            _cc_trimmed = apply_trim(_cc_raw, trim_method, **trim_kw)
            canonical_end         = _cc_trimmed.index[-1]
        _, _, _, canonical_train_split, canonical_val_split = split_timeseries(_cc_trimmed, cfg)
        tqdm.write(f"[{dataset_name}] trim_end={canonical_end.date()}  "
                   f"train_split={canonical_train_split.date()}  val_split={canonical_val_split.date()}")

        series_bar = tqdm(series_map.items(), desc="  Series", unit="series",
                          total=len(series_map), leave=False)
        for series_name, series_fn in series_bar:
            series_bar.set_description(f"  {dataset_name} | {series_name}")
            try:
                raw     = series_fn(log)
                trimmed = raw[raw.index <= canonical_end]
                train, val, test = split_by_dates(trimmed, canonical_train_split, canonical_val_split)
                tqdm.write(f"  {dataset_name}/{series_name}: "
                           f"trimmed {len(raw)}→{len(trimmed)}, "
                           f"train={len(train)} val={len(val)} test={len(test)}")

                if save_plots:
                    plot_timeseries_splits(
                        raw, trimmed, train, val, test,
                        title=f"{dataset_name} — {series_name}",
                        save_path=out_dir / f"{series_name}_splits.png",
                    )

                preds, timing_df = run_pipeline(
                    train, val, test,
                    label=f"{dataset_name}/{series_name}",
                    models=models_to_run,
                    tuning=tuning,
                )
                timing_df["dataset"] = dataset_name
                timing_df["series"]  = series_name
                dataset_timing.append(timing_df)

                if save_plots:
                    plot_forecasts(
                        train, val, test, preds,
                        title=f"{dataset_name} — {series_name}",
                        save_path=out_dir / f"{series_name}.png",
                    )

                y_test = test.to_numpy()
                for model_name, yhat in preds.items():
                    row = dict(
                        dataset=dataset_name, series=series_name,
                        model=model_name,
                        mse=mean_squared_error(y_test, yhat),
                        mae=mean_absolute_error(y_test, yhat),
                    )
                    dataset_rows.append(row)
                    all_rows.append(row)

                    try:
                        pred_by_date = {str(d.date()): float(v) for d, v in zip(test.index, yhat)}
                        _write_ts_prediction_series(model_name, dataset_name, trim_name, True, series_name, pred_by_date)
                    except Exception:
                        tqdm.write(f"  [ts_predictions SKIP] {dataset_name}/{series_name}/{model_name}:\n{traceback.format_exc()}")

            except Exception:
                tqdm.write(f"  [SKIP] {dataset_name}/{series_name}:\n{traceback.format_exc()}")
                continue

        if dataset_rows:
            pd.DataFrame(dataset_rows).to_csv(out_dir / "metrics.csv", index=False)
            if dataset_timing:
                pd.concat(dataset_timing, ignore_index=True).to_csv(out_dir / "time.csv", index=False)
                all_timing.extend(dataset_timing)
            tqdm.write(f"  Saved → {out_dir}/metrics.csv + time.csv" +
                       (f" + *_splits.png + *.png" if save_plots else ""))

        vm_path = out_dir / f"metrics_{dataset_name}_val_mean.csv"
        if not vm_path.exists():
            vm_rows = []
            for series_name, series_fn in series_map.items():
                try:
                    raw     = series_fn(log)
                    trimmed = raw[raw.index <= canonical_end]
                    train, val, test = split_by_dates(trimmed, canonical_train_split, canonical_val_split)
                    yhat   = np.full(len(test), float(val.mean()))
                    y_test = test.to_numpy()
                    vm_rows.append(dict(
                        dataset=dataset_name, series=series_name, model="val_mean",
                        mse=mean_squared_error(y_test, yhat),
                        mae=mean_absolute_error(y_test, yhat),
                    ))
                except Exception:
                    tqdm.write(f"  [val_mean SKIP] {dataset_name}/{series_name}:\n{traceback.format_exc()}")
            if vm_rows:
                pd.DataFrame(vm_rows).to_csv(vm_path, index=False)
                tqdm.write(f"  Saved val_mean → {vm_path}")

    dataset_bar.set_description("Datasets")
    results_df = pd.DataFrame(all_rows)
    if not results_df.empty:
        results_df.to_csv(results_dir / "metrics_all.csv", index=False)
        if all_timing:
            pd.concat(all_timing, ignore_index=True).to_csv(results_dir / "time_all.csv", index=False)
        tqdm.write(f"\nSaved combined → {results_dir}/metrics_all.csv + time_all.csv")

    return results_df

In [ ]:
for trim_cfg in REAL_TRIM_CONFIGS:
    evaluate_all_datasets(
        data_dir    = ROOT / "data" / "real-life",
        results_dir = ROOT / "results",
        tuning      = "small",
        **trim_cfg,
    )

## Val. Mean baseline (Validation Average)

In [ ]:
SERIES_MAP = {
    "concurrent_cases": create_concurrent_cases_timeseries,
    "throughput_time":  create_avg_throughtput_time_timeseries,
}

def collect_val_mean(
    xes_files, log_group, trim_method=None,
    trim_pct=0.25, trim_k=1.5, trim_frac=0.70, trim_window=7,
    train_frac=0.7, val_frac=0.1, test_frac=0.2, cut_date=None,
):
    """Return a list of val_mean result dicts for all XES files and series.

    Each row: {log_group, trim, dataset, series, model, mse, mae}
    """
    cfg     = Split3WayConfig(train_frac=train_frac, val_frac=val_frac, test_frac=test_frac)
    trim_kw = dict(pct=trim_pct, k=trim_k, frac=trim_frac, window=trim_window)
    trim    = _trim_dir_name(trim_method, trim_pct, trim_k, trim_frac)
    kw      = dict(cut_date=cut_date) if cut_date else {}
    rows    = []

    for xes_path in tqdm(xes_files, desc=f"{log_group}/{trim}"):
        dataset_name = Path(xes_path).stem
        try:
            log = pm4py.read_xes(str(xes_path))
        except Exception:
            tqdm.write(f"  [{dataset_name}] FAILED to load:\n{traceback.format_exc()}")
            continue

        _cc_raw     = create_concurrent_cases_timeseries(log, plot=False, **kw)
        if trim_method == "ssd":
            from ssd_trim import run_ssd_trim
            _ssd_result = run_ssd_trim(log, window_step="D")
            canonical_end = _ssd_result["cutoff"] if _ssd_result["cutoff"] is not None else _cc_raw.index[-1]
            _cc_trimmed = _cc_raw[_cc_raw.index <= canonical_end]
        else:
            _cc_trimmed = apply_trim(_cc_raw, trim_method, **trim_kw)
            canonical_end = _cc_trimmed.index[-1]
        _, _, _, canonical_train_split, canonical_val_split = split_timeseries(_cc_trimmed, cfg)

        for series_name, series_fn in SERIES_MAP.items():
            try:
                raw     = series_fn(log, plot=False, **kw)
                trimmed = raw[raw.index <= canonical_end]
                train, val, test = split_by_dates(trimmed, canonical_train_split, canonical_val_split)
                yhat   = np.full(len(test), float(val.mean()))
                y_test = test.to_numpy()
                rows.append(dict(
                    log_group=log_group, trim=trim,
                    dataset=dataset_name, series=series_name, model="val_mean",
                    mse=mean_squared_error(y_test, yhat),
                    mae=mean_absolute_error(y_test, yhat),
                ))
            except Exception:
                tqdm.write(f"  [{dataset_name}/{series_name}] FAILED:\n{traceback.format_exc()}")

    return rows


def save_val_mean_rows(rows, log_group):
    """Merge-safe save: replace only this log_group's rows in results/val_mean_all.csv."""
    new_df = pd.DataFrame(rows)
    vm_path = RESULTS / "val_mean_all.csv"
    if vm_path.exists():
        existing = pd.read_csv(vm_path)
        existing = existing[existing["log_group"] != log_group]
        combined = pd.concat([existing, new_df], ignore_index=True)
    else:
        combined = new_df
    combined.to_csv(vm_path, index=False)
    print(f"Saved → {vm_path}  ({len(new_df)} {log_group} rows, {len(combined)} total)")
    return combined


In [ ]:
real_files = sorted((ROOT / "data" / "real-life").glob("*.xes"))


val_mean_rows = []
for trim_cfg in REAL_TRIM_CONFIGS:
    val_mean_rows += collect_val_mean(xes_files=real_files, log_group="real", **trim_cfg)

print(f"Collected {len(val_mean_rows)} val_mean rows (real-life)")
save_val_mean_rows(val_mean_rows, log_group="real")

## Prophet baseline — batch

In [ ]:
from prophet_baseline import run_prophet_baseline

REAL_LOGS  = [p.stem for p in sorted((ROOT / "data" / "real-life").glob("*.xes"))]
REAL_TRIMS = ["ssd", "none", "peak_0.6", "peak_0.7", "peak_0.8",
              "magnitude_1", "magnitude_2", "magnitude_3"]

jobs = [(ds, trim) for ds in REAL_LOGS for trim in REAL_TRIMS]
print(f"{len(jobs)} jobs  ({len(REAL_LOGS)} real-life datasets × {len(REAL_TRIMS)} trims)")

prophet_summary = []
for ds, trim in jobs:
    xes = ROOT / "data" / "real-life" / f"{ds}.xes"
    if not xes.exists():
        print(f"  [skip] {ds} / {trim} — XES not found"); continue
    try:
        rows = run_prophet_baseline(ds, xes_path=xes, trim=trim, is_real=True, overwrite=True)
        cc_mae = rows[rows["series"] == "concurrent_cases"]["mae"].iloc[0]
        tt_mae = rows[rows["series"] == "throughput_time"]["mae"].iloc[0]
        print(f"  [ok]   {ds:<22} / {trim:<14}  CC MAE={cc_mae:.3f}  TT MAE={tt_mae:.3f}")
        prophet_summary.append(dict(dataset=ds, trim=trim, status="ok", cc_mae=cc_mae, tt_mae=tt_mae))
    except Exception as e:
        print(f"  [ERR]  {ds} / {trim}: {e}")
        prophet_summary.append(dict(dataset=ds, trim=trim, status="error", error=str(e)))

done = sum(1 for s in prophet_summary if s["status"] == "ok")
errs = sum(1 for s in prophet_summary if s["status"] == "error")
print(f"\nDone: {done}  Errors: {errs}")
